# Modules and Model Construction

The networks we have trained so far were small enough to write down layer by
layer. The networks ahead are not: deep ResNets chain more than a hundred
convolutional layers [@He.Zhang.Ren.ea.2016], and a GPT-style language
model stacks dozens of identical Transformer blocks
[@Radford.Wu.Child.ea.2019]. Such models are assembled from repeated
*blocks*, groups of layers treated as units of design. The abstraction that
makes blocks composable is the *module*.

A module is an object with three responsibilities: it owns *parameters*, it
owns *child modules*, and it implements a *forward computation* that maps
inputs to outputs. The definition is deliberately recursive. A fully connected
layer is a module (parameters, no children). A residual block is a module (no
direct parameters, a few child layers that own parameters). A hundred-layer
network is a module whose children are blocks whose children are layers. Most
models therefore have a tree-shaped module hierarchy, as sketched in
the figure. Reusing one child at several sites turns that tree into
an object graph, a case we meet when tying parameters in
that section. Almost everything
this chapter does to a model, listing its parameters
(that section), moving it to a GPU (that section),
saving it to disk (that section), is implemented as a walk over
that tree.

![Layers compose into blocks and blocks compose into models, giving the usual tree-shaped module hierarchy.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-module-tree.svg)

In [ ]:
from dataclasses import dataclass
import tensorflow as tf

## The Module Abstraction

In TensorFlow the module class is Keras's `tf.keras.Model`. We have used one
of its subclasses all along: `tf.keras.Sequential` builds a model from a list
of layers, here the familiar MLP with a 256-unit ReLU hidden layer and a
10-unit output layer. Keras attaches the activation to the `Dense` layer
itself rather than interposing a separate activation layer.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])

X = tf.random.uniform((2, 20))
net(X).shape

`Sequential` is not a special construct. It is itself a model whose forward
computation runs its children in order, and the children are the two layers
we passed to it, held in a tracked list:

In [ ]:
net.layers

This list is what the Keras machinery traverses: `net.trainable_variables`
collects variables by walking the children recursively, and the same walk
underlies serialization. A layer the tracker does not see might as well not
exist, though as we will see shortly, Keras goes to some lengths to make
sure that cannot happen by accident.

In [ ]:
class MLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.hidden = tf.keras.layers.Dense(256, activation='relu')
        self.out = tf.keras.layers.Dense(10)

    def call(self, X):
        return self.out(self.hidden(X))

In [ ]:
net = MLP()
net(X).shape

Two details do the work here. First,
`self.hidden = tf.keras.layers.Dense(...)` is not an ordinary attribute
assignment: Keras intercepts `__setattr__`, sees that the value is a layer,
and adds it to the tracked children we just inspected. That is why both
layers' variables show up in `net.trainable_variables` with no further
ceremony. Second, we never wrote a backward method; automatic differentiation
derives gradients from whatever `call` computes.

## Sequential and Friends: Containers

To see that there is no magic left in `Sequential`, we can write it
ourselves. Two ingredients suffice: store the children in an attribute, and
loop over them in `call`.

In [ ]:
class MySequential(tf.keras.Model):
    def __init__(self, *args):
        super().__init__()
        self.modules = args

    def call(self, X):
        for module in self.modules:
            X = module(X)
        return X

`self.modules = args` looks like an ordinary assignment, and that is the
point: the `__setattr__` interception scans whatever is assigned, looking
inside lists, tuples, and dictionaries, so both `Dense` children are tracked
and appear in `net.layers`. Our version is a drop-in replacement:

In [ ]:
net = MySequential(tf.keras.layers.Dense(256, activation='relu'),
                   tf.keras.layers.Dense(10))
net(X).shape

In older imperative frameworks this registration step is famously easy to
lose: store the children in a plain Python list instead of the framework's
dedicated container, and their parameters silently vanish from the model.
Keras closes that trap. Because the attribute scan looks inside lists and
dictionaries, a plain list of layers is tracked like any other child:

In [ ]:
class ListMLP(tf.keras.Model):
    def __init__(self):
        super().__init__()
        self.blocks = [tf.keras.layers.Dense(256, activation='relu'),
                       tf.keras.layers.Dense(10)]

    def call(self, X):
        for block in self.blocks:
            X = block(X)
        return X

net = ListMLP()
net(X).shape, sum(int(tf.size(v)) for v in net.trainable_variables)

All 7946 parameters are present; there is no broken variant of this model to
show. (We named the list `blocks` only because `layers` is reserved: Keras
maintains that attribute itself, and assigning to it raises an error.) The
structural mistake Keras does guard against is different in kind and, more
usefully, in loudness. Building is a commitment: once a model's variables
exist, its set of children is locked, and attaching a new layer to a built
model fails at the assignment itself:

In [ ]:
try:
    net.head = tf.keras.layers.Dense(2)  # net is built: too late
except ValueError as e:
    print(e)

The error message states the rule Keras enforces: all state must be created
in `__init__` or in `build`, never after. Keras therefore ships no special
list or dict containers, because none are needed: any layer reachable from an
attribute is tracked, and a late structural edit fails at assignment time
rather than being silently ignored. Transformer implementations in Keras
conventionally keep their stack of blocks in exactly the kind of plain list
`ListMLP` used, with the named parts (embedding, final normalization, output
head) as separate attributes.

## Forward Is Just Python

`call` is an ordinary Python method. Nothing restricts it to chaining
children: it can branch, loop, call any TensorFlow function, and combine
intermediate results however it likes; TensorFlow executes eagerly, so all of
this runs one operation at a time, just as in NumPy. (Once a model is wrapped
in `tf.function` for speed, as the `Trainer` from that section
does, AutoGraph rewrites such control flow into graph form.) The loop in
`MySequential` already used this freedom. Its most consequential one-line use
is the *residual connection*, the wiring idiom at the heart of ResNets and
Transformers alike:

In [ ]:
class ResidualBlock(tf.keras.Model):
    def __init__(self, num_hiddens):
        super().__init__()
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

![The residual wiring `X + body(X)`: the input splits at a branch point into the body stack and an identity skip, and the two rejoin by addition before the block's output.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-residual-block.svg)

`X + body(X)` is not a layer any framework provides. It is plain arithmetic
in the forward computation, and it changes what the block *is*: the block
computes a perturbation of the identity function rather than an arbitrary
transformation. If its body is $F$, its Jacobian is $I + J_F$, so
backpropagation receives an additive identity contribution along the skip
path. This contribution can still cancel against $J_F$; it is a direct path,
not a guarantee that every gradient is preserved
[@He.Zhang.Ren.ea.2016]. the figure diagrams
exactly this wiring, and that section develops both points when we
build ResNet; for now we only need the mechanics. One mechanical consequence
is visible already: the addition forces the input and output shapes to
agree, so a residual block has a single width that is part of its identity.

Keras always infers input widths at build time, so here the constraint falls
on the caller: feed the block anything other than `num_hiddens` columns and
the addition fails.

In [ ]:
block = ResidualBlock(24)
block(tf.random.normal((2, 24))).shape

`call` may also use state that is neither an input nor a parameter. Suppose
we want to damp each block's contribution by a fixed factor:

In [ ]:
class ScaledResidual(ResidualBlock):
    def __init__(self, num_hiddens, alpha=0.5):
        super().__init__(num_hiddens)
        self.alpha = tf.constant(alpha)  # Fixed by design, never trained

    def call(self, X):
        return X + self.alpha * self.body(X)

block = ScaledResidual(24)
block(tf.random.normal((2, 24)))
any('alpha' in w.path for w in block.weights), [w.path for w in block.weights][:2]

`alpha` enters the computation, but it is not a parameter: as the output
shows, `block.weights` contains only the `Dense` variables, so the optimizer
never touches it. That much we wanted. Storing it as a `tf.constant` has a
cost we did not want, though: not being a variable, it is invisible to weight
checkpointing as well, so it will not be saved with the model. Some state is
not a parameter but must still travel with the model; the registered home for
such state is a non-trainable `tf.Variable`, introduced in
that section.

## Lazy Initialization: Shapes from Data

We have been doing something odd since our first MLP without commenting on
it: `Dense(256)` names only the layer's *output* width. Its kernel has shape
`(in_features, 256)`, and we never said what `in_features` is. The layer
cannot know it at construction time, since it depends on the data it will
receive. So Keras never allocates variables at construction time: every layer
has a `build(input_shape)` method that creates them, and `__call__` invokes
it the first time data arrives. Deferred building is not a special mode; it
is how every Keras layer works.

In [ ]:
net = tf.keras.Sequential([tf.keras.layers.Dense(256, activation='relu'),
                           tf.keras.layers.Dense(10)])
net.weights

There are no variables at all yet, and accessing `net.layers[0].kernel` would
raise an error telling us to build the layer first. The first time data flows
through, each layer's `build` reads the input width from the incoming shape
and allocates and initializes a real kernel, and the layer's declared output
width fixes the input of the next layer, so shapes cascade through the whole
model:

In [ ]:
net(X)
[w.shape for w in net.weights]

Lazy layers can remove shape arithmetic from model definitions. In a
convolutional network the flattened feature-map size depends on the input
resolution and upstream strides. We usually avoid that fragile arithmetic by
using global or adaptive pooling before the first linear layer. When a width
is part of the architecture, we name it in the model config and pass it to
adjacent layers — code that stays explicit and compact without hiding
parameter creation behind a sample batch, whether or not the framework
offers a lazy mode.

Until the first call, the variables do not exist. An optimizer object can be
constructed without them, but reading weights, counting parameters, or
building optimizer state requires the model's variable list. Keras offers two
ways to create it: a dry run on a
representative batch, or `net.build((None, 20))`, which propagates shapes
through the model without any data (`None` marks the batch dimension).
Initialization
happens at build rather than at construction, so under a fixed seed the
weights you get depend on how many random numbers the program drew before the
model was built (that section returns to seeding). As the containers
lesson showed, building is also the moment the model's structure locks.

In [ ]:
net = tf.keras.Sequential([
    tf.keras.layers.Dense(
        256, activation='relu',
        kernel_initializer=tf.keras.initializers.GlorotUniform()),
    tf.keras.layers.Dense(10)])
net.build((None, 20))
net.layers[0].kernel.shape

`build` materialized every shape and drew the first kernel's initial values
from the Xavier initializer, without a batch of data in sight. Which
initializer to use, and why Xavier's variance rule (that section)
is a sensible default, is the subject of that section.

## Building from a Config

So far every width in this section was a literal typed into a constructor.
Real model code does not work that way. The architecture choices that vary
across experiments, such as depth, width, and output size, must be logged with
results and stored with checkpoints so that a saved model can be rebuilt. The
topology remains in code; a small configuration object records its variable
choices:

In [ ]:
@dataclass
class MLPConfig:
    d_in: int = 784
    d_hidden: int = 256
    num_blocks: int = 4
    d_out: int = 10

def build(cfg: MLPConfig) -> tf.keras.Model:
    blocks = [ResidualBlock(cfg.d_hidden) for _ in range(cfg.num_blocks)]
    return tf.keras.Sequential([tf.keras.Input((cfg.d_in,)),
                                tf.keras.layers.Dense(cfg.d_hidden),
                                *blocks,
                                tf.keras.layers.Dense(cfg.d_out)])

One config produces one architecture: an input projection into the hidden
width, `num_blocks` identical residual blocks, and an output projection. The
free-standing `build(cfg)` function is unrelated to the `build` method Keras
calls on layers, but the two meet here: because the config knows the input
width, we can declare it with `tf.keras.Input`, and `Sequential` then builds
the whole model at construction time, no dry run needed. This is where
explicit shapes beat inferred ones. `net.summary()` displays the module tree,
every shape and parameter count already known:

In [ ]:
net = build(MLPConfig())
net.summary()

In [ ]:
net(tf.random.uniform((2, 784))).shape

The variable architecture choices are now data. Rescaling the model is a change to two fields, not
an edit to model code:

In [ ]:
for cfg in (MLPConfig(), MLPConfig(d_hidden=512, num_blocks=8)):
    print(f'd_hidden={cfg.d_hidden}, num_blocks={cfg.num_blocks}: '
          f'{build(cfg).count_params():,} parameters')

Because `build` is deterministic in `cfg`, the config is all you need to
reconstruct the module tree later; that section saves it
alongside the weights so that loading a checkpoint starts by rebuilding the
exact same model. Keras bakes the same idea into every layer: `get_config()`
returns the constructor arguments needed to re-create the object, and that is
what Keras model serialization records. A config of widths and depths feeding
a loop that stacks identical residual blocks is a common construction pattern
in the Transformer implementations later in the book.

## Summary

A module owns variables, child layers, and a `call` method. Layers, blocks,
and whole models are the same kind of object, so the usual hierarchy is
tree-shaped, while shared children introduce aliases. Variable collection and
serialization traverse that object graph. Children are discovered through
attribute assignment: Keras scans every
assigned attribute, lists and dictionaries included, so even a plain Python
list of layers is tracked. Variables are created by `build`, not by the
constructor: every layer infers its input width when the first batch arrives
(or when `build(input_shape)` is called), after which the model's structure
is locked. `call` is ordinary Python; a residual connection is one line in
it. Configs turn architecture into data: a `dataclass` of widths and depths
plus a `build(cfg)` function that stacks blocks, with `get_config()` as the
Keras-native form of the same idea.

## Exercises

1. Keras's tracking has one blind spot left: create a `Dense` layer *inside*
   `call` rather than in the constructor. The model runs without complaint;
   check `len(net.trainable_variables)` after calling it, explain what an
   optimizer would train, and explain what happens to the layer's weights
   between two calls.
1. Implement a `ParallelBlock` that takes two child modules `net1` and `net2`,
   runs both on the same input, and concatenates their outputs along the last
   dimension. What must be true of the two children's outputs for the
   concatenation to be valid?
1. Extend `MLPConfig` with an activation switch (for example,
   `act: str = 'relu'`) and make `build` honor it. Which decisions belong in a
   config and which belong in code? Where would you put a choice between
   `ResidualBlock` and a plain feed-forward block?
1. `ResidualBlock` requires its input and output widths to agree. Suppose you
   want a block whose output is wider than its input. Give two standard fixes
   and the cost of each. (that section uses one of them in ResNet.)